In [6]:
import pandas as pd
import pyodbc
import re

In [7]:
Connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

Con = pyodbc.connect(Connection_string)

In [8]:
query = """
SELECT CLICodigo,
       CLINombres,
       CLIApellidos,
       CLIFechaNacimiento,
       CLIEmailPrincipal,
       CLICelular,
       CLIUltTrx
FROM Clientes
WHERE MONTH(CLIFechaNacimiento) BETWEEN 1 AND 5
"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, Con)
df.head()


C:\Users\Asus\AppData\Local\Temp\ipykernel_18516\9271266.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, Con)


,CLICodigo,CLINombres,CLIApellidos,CLIFechaNacimiento,CLIEmailPrincipal,CLICelular,CLIUltTrx
0,1013260820,MARIANA,DIAZ,2023-01-19 00:00:00,MARIDIAZD08@HOTMAIL.COM,3134495621,2023-01-15
1,1017218266,DIEGO,REYES GONZALEZ,1994-04-11 00:00:00,DIREYESGO30@GMAIL.COM,3052270509,2021-06-06
2,1019024338,MONICA PAOLA,GARZON GIL,1988-03-25 00:00:00,MONISOFIA2014@GMAIL.COM,3144874565,2021-06-09
3,1020742092,ALEXANDRA,PIÑEROS PINEDA,1989-02-25 00:00:00,MAYITO0225TWILIGHT@HOTAIL.COM,3194843765,2021-06-03
4,1035231968,LIZET DAYANA,PINTO RESTREPO,1996-01-23 00:00:00,LIZETH1623@HOTMAIL.COM,3105047389,2021-06-04


In [9]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip().lower())

def es_email_valido(email):
    email = str(email).strip().lower()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["negad", "pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [10]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [11]:
# Exportar todos los registros con validaciones incluidas
df.to_excel("clientes_con_validacion3.xlsx", index=False)